# 🧭 验证 show_navigation_proposal → action_candidate

**目的：** 验证 agent 在注入 `show_navigation_proposal` 前端工具后，能否正确返回 `action_candidate`。

**背景问题：** 真实请求中 `tools[]` 有 `show_navigation_proposal`，但 agent 读的是
`state['copilotkit']['actions']`。若 state 没注入这个字段，LLM 看不到该工具，不会调用。

**验证矩阵：**

| 场景 | 用户消息 | 期望行为 |
|------|----------|----------|
| A | 当前有哪些项目 | 调用 `list_projects` 返回数据（非导航） |
| B | 帮我去数据模型管理 | 调用 `show_navigation_proposal` 返回 `action_candidate` |
| C | 帮我配置权限 | 调用 `show_navigation_proposal` 返回 `action_candidate` |
| D | 项目 | 意图模糊，返回 `clarification_candidate` |


## 0. 环境初始化

In [1]:
import sys, os, json, uuid
from unittest.mock import AsyncMock, patch, MagicMock

AGENT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(AGENT_ROOT, '.env'))

from logging_setup import setup_logging
setup_logging()

import config
print('AGENT_ROOT:', AGENT_ROOT)
print('LLM_MODEL :', config.LLM_MODEL)

AGENT_ROOT: /data/home/lukemxjia/modelcraft/modelcraft-agent
LLM_MODEL : deepseek-chat


## 1. 前端工具定义（模拟 CopilotKit 注入的 actions）

`agent_node` 从 `state['copilotkit']['actions']` 读取前端工具，
这里按真实请求 payload 原样构造。

In [2]:
# 与真实请求 body.tools[] 保持一致
SHOW_TOAST_TOOL = {
    "name": "show_toast",
    "description": "向用户显示一条临时通知消息（不需要用户在聊天框内查看）",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "通知内容"},
            "type": {"type": "string", "description": "success | error | info | warning（默认 info）"}
        },
        "required": ["message"]
    }
}

SHOW_NAVIGATION_PROPOSAL_TOOL = {
    "name": "show_navigation_proposal",
    "description": "向用户展示 AI 导航提案卡片，用户可点击候选项执行页面跳转、元素高亮或发送澄清消息",
    "parameters": {
        "type": "object",
        "properties": {
            "response": {
                "type": "object",
                "properties": {}
            }
        },
        "required": ["response"]
    }
}

# 真实请求中的 routeCatalog context
ROUTE_CATALOG = [
    {"routeTemplate": "/org/:orgName/workspace", "title": "项目列表", "description": "查看、搜索、创建和管理组织下的所有项目", "keywords": ["项目列表", "所有项目", "workspace"]},
    {"routeTemplate": "/org/:orgName/project/:projectSlug/model-editor", "title": "数据模型编辑器", "description": "创建和管理数据模型、字段结构", "keywords": ["模型", "字段", "数据模型", "模型编辑器"]},
    {"routeTemplate": "/org/:orgName/project/:projectSlug/rbac/roles", "title": "RBAC 角色管理", "description": "管理项目内的角色与权限包", "keywords": ["权限", "RBAC", "角色"]},
    {"routeTemplate": "/org/:orgName/project/:projectSlug/end-users", "title": "终端用户管理", "description": "管理访问本项目的终端用户账号", "keywords": ["终端用户", "end user"]},
    {"routeTemplate": "/org/:orgName/project/:projectSlug/settings", "title": "项目设置", "description": "修改项目基本信息、管理数据库集群", "keywords": ["项目设置", "集群配置", "数据库连接"]},
]

AI_TARGETS = [
    {"id": "create_project", "label": "新建项目", "description": "点击打开新建项目表单"}
]

print('✅ 前端工具定义完成')
print(f'  - show_toast')
print(f'  - show_navigation_proposal')
print(f'  routeCatalog: {len(ROUTE_CATALOG)} 条')
print(f'  aiTargets: {len(AI_TARGETS)} 条')

✅ 前端工具定义完成
  - show_toast
  - show_navigation_proposal
  routeCatalog: 5 条
  aiTargets: 1 条


## 2. Mock GraphQL Client

In [3]:
FAKE_PROJECTS = [
    {"id": "p1", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE"},
    {"id": "p2", "slug": "abcde",        "title": "abcde",       "description": "", "status": "ACTIVE"},
    {"id": "p3", "slug": "pro1",         "title": "pro1",        "description": "", "status": "ACTIVE"},
]

def make_fake_client(state=None):
    c = MagicMock()
    c.list_projects    = AsyncMock(return_value={"data": {"projects": FAKE_PROJECTS}})
    c.list_databases   = AsyncMock(return_value={"data": {"listDatabases": {"edges": []}}}  )
    c.list_models      = AsyncMock(return_value={"data": {"models": {"items": [], "hasNextPage": False}}})
    c.get_model_fields = AsyncMock(return_value={"data": {"model": {"fields": []}}})
    c.query_model      = AsyncMock(return_value={"data": {"queryModel": {"items": [], "hasNextPage": False}}})
    c.nl2filter        = AsyncMock(return_value={"data": {"nl2filter": {"filter": "{}", "explanation": "mock"}}})
    return c

# ⚠️  tools.py 做的是 `from agents.shared import make_client`，
#    所以必须 patch agents.tools.make_client（本地引用），
#    而非 agents.shared.make_client（patch 了也不影响已经绑定的名字）
PATCH_TARGET = 'agents.tools.make_client'
print(f'✅ mock 就绪，patch target = {PATCH_TARGET}')

✅ mock 就绪，patch target = agents.tools.make_client


## 3. State 构造函数

**关键：** 注入 `copilotkit.actions`，让 agent_node 能把前端工具绑到 LLM。

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

ORG_NAME     = "lukeco"
PROJECT_SLUG = "onboardceshi"
LAYER        = "org"

# routeCatalog 注入为 SystemMessage（模拟 CopilotKit useCopilotReadable 的效果）
ROUTE_CATALOG_MSG = SystemMessage(content=(
    "系统所有可导航页面目录（routeCatalog）。"
    "调用 show_navigation_proposal 时，ui.navigate 的 route 字段必须从 routeTemplate 派生，"
    "将 :orgName 替换为实际 orgName，:projectSlug 替换为实际 projectSlug。\n"
    + json.dumps(ROUTE_CATALOG, ensure_ascii=False)
))

AI_TARGETS_MSG = SystemMessage(content=(
    "当前页面已注册的 AI 高亮目标（aiTargets）。"
    "调用 show_navigation_proposal 时，ui.highlight 的 targetId 必须从这个列表中选取。\n"
    + json.dumps(AI_TARGETS, ensure_ascii=False)
))

def make_state(message: str, inject_frontend_tools: bool = True) -> dict:
    """
    inject_frontend_tools=True 时模拟真实 CopilotKit 请求（含 show_navigation_proposal）。
    routeCatalog 和 aiTargets 作为 SystemMessage 注入 messages，模拟 useCopilotReadable 效果。
    """
    state: dict = {
        "messages"     : [ROUTE_CATALOG_MSG, AI_TARGETS_MSG, HumanMessage(content=message)],
        "authorization": "Bearer fake-offline-token",
        "org_name"     : ORG_NAME,
        "layer"        : LAYER,
        "project_slug" : PROJECT_SLUG,
        "current_model": "",
        "current_db"   : "",
    }
    if inject_frontend_tools:
        state["copilotkit"] = {
            "actions": [SHOW_TOAST_TOOL, SHOW_NAVIGATION_PROPOSAL_TOOL],
        }
    return state

def new_cfg() -> dict:
    tid = str(uuid.uuid4())
    return {"configurable": {"thread_id": tid}}

# 每次测试前重置 LazyGraph，让 graph 在 mock 上下文内重新编译
def reset_graph():
    from agents import admin_agent
    admin_agent._LazyGraph._instance = None

print('✅ make_state 已定义（routeCatalog + aiTargets 注入为 SystemMessage）')
print('✅ reset_graph 已定义（每次测试前调用，确保 mock 生效）')

✅ make_state 已定义（routeCatalog + aiTargets 注入为 SystemMessage）
✅ reset_graph 已定义（每次测试前调用，确保 mock 生效）


## 4. 工具函数：打印 tool_calls 详情

In [5]:
from langchain_core.messages import AIMessage, ToolMessage

def _parse_response(raw) -> dict:
    """response 可能是 dict 或 JSON 字符串，统一转成 dict。"""
    if isinstance(raw, dict):
        return raw
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except Exception:
            return {"_raw": raw}
    return {}

def inspect_result(result: dict, label: str = ""):
    """打印消息链，重点展示 tool_calls 和 show_navigation_proposal 的 response 参数。"""
    msgs = result["messages"]
    print(f"\n{'='*60}")
    if label:
        print(f"  {label}")
    print(f"  共 {len(msgs)} 条消息")
    print('='*60)

    for i, m in enumerate(msgs):
        role = type(m).__name__
        if isinstance(m, AIMessage) and getattr(m, 'tool_calls', None):
            for tc in m.tool_calls:
                name = tc['name']
                args = tc.get('args', {})
                if name == 'show_navigation_proposal':
                    raw_resp = args.get('response', {})
                    print(f"  [{i}] AIMessage → 🧭 show_navigation_proposal")
                    print(f"       response type: {type(raw_resp).__name__}  ← {'✅ dict' if isinstance(raw_resp, dict) else '⚠️  string (LLM 序列化问题)'}")
                    resp = _parse_response(raw_resp)
                    candidates = resp.get('candidates', [])
                    print(f"       message     : {resp.get('message', '')}")
                    print(f"       proposalType: {resp.get('proposalType', '')}")
                    print(f"       candidates ({len(candidates)}):")
                    for c in candidates:
                        ctype = c.get('type', '?')
                        ctitle = c.get('title', '')
                        actions = c.get('actions', [])
                        print(f"         {'✅' if ctype == 'action_candidate' else '❓'} [{ctype}] {ctitle}")
                        for a in actions:
                            print(f"              {a.get('type')} → {json.dumps(a.get('args', {}), ensure_ascii=False)}")
                else:
                    print(f"  [{i}] AIMessage → 🔧 {name}({json.dumps(args, ensure_ascii=False)[:80]})")
        elif isinstance(m, ToolMessage):
            print(f"  [{i}] ToolMessage  → {str(m.content)[:80]}")
        elif isinstance(m, AIMessage):
            print(f"  [{i}] AIMessage  text={str(m.content)[:80]}")
        else:
            print(f"  [{i}] {role}  {str(m.content)[:80]}")

print('✅ inspect_result 已定义（支持 response 为 string 的情况）')

✅ inspect_result 已定义（支持 response 为 string 的情况）


## 5. 场景 A：数据查询「当前有哪些项目」

**期望：** 调用 `list_projects` 返回数据，不调用 `show_navigation_proposal`

In [6]:
from agents.admin_agent import admin_graph

reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_a = await admin_graph.ainvoke(
        make_state("当前有哪些项目"),
        config=new_cfg()
    )

inspect_result(result_a, "场景 A：当前有哪些项目")

{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "show_navigation_proposal"], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:13.237630Z"}


{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-22T09:02:14.592299Z"}


{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 3.72, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-22T09:02:14.596488Z"}


{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "show_navigation_proposal"], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:14.597851Z"}



  场景 A：当前有哪些项目
  共 6 条消息
  [0] SystemMessage  系统所有可导航页面目录（routeCatalog）。调用 show_navigation_proposal 时，ui.navigate 的 route 字段必须
  [1] SystemMessage  当前页面已注册的 AI 高亮目标（aiTargets）。调用 show_navigation_proposal 时，ui.highlight 的 targetI
  [2] HumanMessage  当前有哪些项目
  [3] AIMessage → 🔧 list_projects({})
  [4] ToolMessage  → [{"id": "p1", "slug": "onboardceshi", "title": "onboard测试", "description": "", "
  [5] AIMessage → 🧭 show_navigation_proposal
       response type: dict  ← ✅ dict
       message     : 找到以下项目：
       proposalType: navigation
       candidates (3):
         ✅ [action_candidate] onboard测试 (onboardceshi)
              ui.navigate → {"route": "/org/lukeco/project/onboardceshi/model-editor"}
         ✅ [action_candidate] abcde
              ui.navigate → {"route": "/org/lukeco/project/abcde/model-editor"}
         ✅ [action_candidate] pro1
              ui.navigate → {"route": "/org/lukeco/project/pro1/model-editor"}


## 6. 场景 B：导航请求「帮我去数据模型管理」

**期望：** 调用 `show_navigation_proposal`，返回 `action_candidate`，
route 为 `/org/lukeco/project/onboardceshi/model-editor`

In [7]:
reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_b = await admin_graph.ainvoke(
        make_state("帮我去数据模型管理"),
        config=new_cfg()
    )

inspect_result(result_b, "场景 B：帮我去数据模型管理")

{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "show_navigation_proposal"], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:17.994809Z"}



  场景 B：帮我去数据模型管理
  共 4 条消息
  [0] SystemMessage  系统所有可导航页面目录（routeCatalog）。调用 show_navigation_proposal 时，ui.navigate 的 route 字段必须
  [1] SystemMessage  当前页面已注册的 AI 高亮目标（aiTargets）。调用 show_navigation_proposal 时，ui.highlight 的 targetI
  [2] HumanMessage  帮我去数据模型管理
  [3] AIMessage → 🧭 show_navigation_proposal
       response type: dict  ← ✅ dict
       message     : 为您找到数据模型编辑器页面，点击即可跳转：
       proposalType: navigation
       candidates (1):
         ✅ [action_candidate] 数据模型编辑器 - onboardceshi
              ui.navigate → {"route": "/org/lukeco/project/onboardceshi/model-editor"}


## 7. 场景 C：导航请求「帮我配置权限」

**期望：** 调用 `show_navigation_proposal`，返回 `action_candidate`，
route 包含 `/rbac/roles`

In [8]:
reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_c = await admin_graph.ainvoke(
        make_state("帮我配置权限"),
        config=new_cfg()
    )

inspect_result(result_c, "场景 C：帮我配置权限")

{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "show_navigation_proposal"], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:20.420061Z"}



  场景 C：帮我配置权限
  共 4 条消息
  [0] SystemMessage  系统所有可导航页面目录（routeCatalog）。调用 show_navigation_proposal 时，ui.navigate 的 route 字段必须
  [1] SystemMessage  当前页面已注册的 AI 高亮目标（aiTargets）。调用 show_navigation_proposal 时，ui.highlight 的 targetI
  [2] HumanMessage  帮我配置权限
  [3] AIMessage → 🧭 show_navigation_proposal
       response type: dict  ← ✅ dict
       message     : 已找到权限配置入口，点击下方按钮跳转到 RBAC 角色管理页面：
       proposalType: navigation
       candidates (1):
         ✅ [action_candidate] RBAC 角色管理
              ui.navigate → {"route": "/org/lukeco/project/onboardceshi/rbac/roles"}


## 8. 场景 D：模糊意图「项目」

**期望：** 返回 `clarification_candidate`（意图不明），或 `action_candidate`（直接跳项目列表）

In [9]:
reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_d = await admin_graph.ainvoke(
        make_state("项目"),
        config=new_cfg()
    )

inspect_result(result_d, "场景 D：模糊意图 '项目'")

{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "show_navigation_proposal"], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:22.865223Z"}


{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-22T09:02:24.061437Z"}


{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 3.34, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-22T09:02:24.065240Z"}


{"service": "modelcraft-agent", "copilotkit_keys": ["actions"], "frontend_action_count": 2, "frontend_action_names": ["show_toast", "show_navigation_proposal"], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:24.066344Z"}



  场景 D：模糊意图 '项目'
  共 6 条消息
  [0] SystemMessage  系统所有可导航页面目录（routeCatalog）。调用 show_navigation_proposal 时，ui.navigate 的 route 字段必须
  [1] SystemMessage  当前页面已注册的 AI 高亮目标（aiTargets）。调用 show_navigation_proposal 时，ui.highlight 的 targetI
  [2] HumanMessage  项目
  [3] AIMessage → 🔧 list_projects({})
  [4] ToolMessage  → [{"id": "p1", "slug": "onboardceshi", "title": "onboard测试", "description": "", "
  [5] AIMessage → 🧭 show_navigation_proposal
       response type: dict  ← ✅ dict
       message     : 找到以下项目，点击可进入对应页面：
       proposalType: navigation
       candidates (3):
         ✅ [action_candidate] 📁 onboard测试 (onboardceshi)
              ui.navigate → {"route": "/org/lukeco/project/onboardceshi/model-editor"}
         ✅ [action_candidate] 📁 abcde
              ui.navigate → {"route": "/org/lukeco/project/abcde/model-editor"}
         ✅ [action_candidate] 📁 pro1
              ui.navigate → {"route": "/org/lukeco/project/pro1/model-editor"}


## 9. 对比：不注入前端工具时的行为

验证 **根本原因**：没有 `copilotkit.actions` 时 LLM 看不到 `show_navigation_proposal`，只会用文字描述

In [10]:
reset_graph()
with patch(PATCH_TARGET, side_effect=make_fake_client):
    result_no_tools = await admin_graph.ainvoke(
        make_state("帮我去数据模型管理", inject_frontend_tools=False),
        config=new_cfg()
    )

print("\n[不注入前端工具] Agent 回复：")
print(result_no_tools['messages'][-1].content)
print("\n↑ 只有文字描述，没有 show_navigation_proposal 调用 ← 这就是 bug 的根本原因")

{"service": "modelcraft-agent", "copilotkit_keys": [], "frontend_action_count": 0, "frontend_action_names": [], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:27.427039Z"}


{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-22T09:02:28.842250Z"}


{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 3.39, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-22T09:02:28.846113Z"}


{"service": "modelcraft-agent", "copilotkit_keys": [], "frontend_action_count": 0, "frontend_action_names": [], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:28.847126Z"}


{"service": "modelcraft-agent", "copilotkit_keys": [], "frontend_action_count": 0, "frontend_action_names": [], "event": "agent_node.debug", "level": "info", "timestamp": "2026-05-22T09:02:31.164045Z"}



[不注入前端工具] Agent 回复：
好的，以下是导航信息：

---

**数据模型编辑器** 页面位于当前项目 **onboardceshi** 下。

您可以通过以下路径手动进入：

1. 在左侧导航栏找到当前项目 **onboardceshi**
2. 点击进入 **数据模型编辑器**（Model Editor）

或者直接访问链接：`/org/lukeco/project/onboardceshi/model-editor

---

需要我帮您进一步查看该项目下的数据库或数据模型吗？

↑ 只有文字描述，没有 show_navigation_proposal 调用 ← 这就是 bug 的根本原因


## 10. 断言验证（自动检查）

In [11]:
def get_tool_calls(result: dict) -> list[dict]:
    calls = []
    for m in result['messages']:
        if isinstance(m, AIMessage) and getattr(m, 'tool_calls', None):
            calls.extend(m.tool_calls)
    return calls

def get_proposal_candidates(result: dict) -> list[dict]:
    for tc in get_tool_calls(result):
        if tc['name'] == 'show_navigation_proposal':
            resp = _parse_response(tc.get('args', {}).get('response', {}))
            return resp.get('candidates', [])
    return []

passed = []
failed = []

def check(name, cond, detail=""):
    if cond:
        passed.append(name)
        print(f"  ✅ {name}")
    else:
        failed.append(name)
        print(f"  ❌ {name}  {detail}")

print("\n── 断言结果 ──────────────────────────────────────")

# A: 「当前有哪些项目」→ list_projects + show_navigation_proposal，每个项目一个 action_candidate
calls_a = [tc['name'] for tc in get_tool_calls(result_a)]
cands_a = get_proposal_candidates(result_a)
action_cands_a = [c for c in cands_a if c.get('type') == 'action_candidate']
check("A: list_projects 被调用", "list_projects" in calls_a, f"实际: {calls_a}")
check("A: show_navigation_proposal 被调用", "show_navigation_proposal" in calls_a, f"实际: {calls_a}")
check("A: 每个项目有 action_candidate", len(action_cands_a) >= 3, f"实际 candidates: {len(action_cands_a)}")
if action_cands_a:
    routes_a = [a.get('args', {}).get('route', '') for c in action_cands_a for a in c.get('actions', []) if a.get('type') == 'ui.navigate']
    check("A: routes 含 model-editor", all('model-editor' in r for r in routes_a if r), f"routes: {routes_a}")

# B: 「帮我去数据模型管理」→ 直接 show_navigation_proposal，无多余工具调用
calls_b = [tc['name'] for tc in get_tool_calls(result_b)]
cands_b = get_proposal_candidates(result_b)
action_cands_b = [c for c in cands_b if c.get('type') == 'action_candidate']
check("B: 直接 show_navigation_proposal（无后端工具）", calls_b == ['show_navigation_proposal'], f"实际: {calls_b}")
check("B: 含 action_candidate", len(action_cands_b) > 0, f"类型: {[c.get('type') for c in cands_b]}")
b_raw = next((tc.get('args', {}).get('response') for tc in get_tool_calls(result_b) if tc['name'] == 'show_navigation_proposal'), None)
check("B: response 是 dict", isinstance(b_raw, dict), f"类型: {type(b_raw).__name__ if b_raw else 'None'}")
if action_cands_b:
    routes_b = [a.get('args', {}).get('route', '') for c in action_cands_b for a in c.get('actions', []) if a.get('type') == 'ui.navigate']
    check("B: route 含 model-editor", any('model-editor' in r for r in routes_b), f"routes: {routes_b}")

# C: 「帮我配置权限」→ 直接 show_navigation_proposal，不调 list_databases
calls_c = [tc['name'] for tc in get_tool_calls(result_c)]
cands_c = get_proposal_candidates(result_c)
action_cands_c = [c for c in cands_c if c.get('type') == 'action_candidate']
check("C: 未调 list_databases（导航意图不查数据）", "list_databases" not in calls_c, f"实际: {calls_c}")
check("C: show_navigation_proposal 被调用", "show_navigation_proposal" in calls_c, f"实际: {calls_c}")
check("C: 含 action_candidate", len(action_cands_c) > 0, f"类型: {[c.get('type') for c in cands_c]}")
if action_cands_c:
    routes_c = [a.get('args', {}).get('route', '') for c in action_cands_c for a in c.get('actions', []) if a.get('type') == 'ui.navigate']
    check("C: route 含 rbac", any('rbac' in r for r in routes_c), f"routes: {routes_c}")

# D: 模糊意图 → 导航候选或数据查询都接受
calls_d = [tc['name'] for tc in get_tool_calls(result_d)]
cands_d = get_proposal_candidates(result_d)
check("D: 导航 or 数据查询", len(cands_d) > 0 or "list_projects" in calls_d,
      f"tool_calls={calls_d}")

print(f"\n{'='*50}")
print(f"  通过: {len(passed)} / 失败: {len(failed)}")
if failed:
    print(f"  ❌ 失败项: {failed}")
else:
    print("  🎉 全部通过")


── 断言结果 ──────────────────────────────────────
  ✅ A: list_projects 被调用
  ✅ A: show_navigation_proposal 被调用
  ✅ A: 每个项目有 action_candidate
  ✅ A: routes 含 model-editor
  ✅ B: 直接 show_navigation_proposal（无后端工具）
  ✅ B: 含 action_candidate
  ✅ B: response 是 dict
  ✅ B: route 含 model-editor
  ✅ C: 未调 list_databases（导航意图不查数据）
  ✅ C: show_navigation_proposal 被调用
  ✅ C: 含 action_candidate
  ✅ C: route 含 rbac
  ✅ D: 导航 or 数据查询

  通过: 13 / 失败: 0
  🎉 全部通过
